In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option('display.max_columns', 50)

#Load Dataset from Kaggle

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'apiKey'

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'titanic-dataset' dataset.
Path to dataset files: /kaggle/input/titanic-dataset


In [ ]:
import shutil
shutil.copytree(
    path,
    'titanic-dataset',
    dirs_exist_ok=True
)

'titanic-dataset'

In [ ]:
df = pd.read_csv("/content/titanic-dataset/Titanic-Dataset.csv")
df.head()
df.shape

(891, 12)

In [ ]:
df.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


#Adding few columns for demonstrating data issues

In [ ]:
df['passenger_id'] = np.arange(1,len(df)+1) #In ML we drop this columns in real cause it does not have any influence on output
#Constant Column -> also no influence in output cause all values are same
df['constant_col'] = 1
#Create a dirty version of embarked_town
df['embarked_town_dirty'] = df['Embarked'].copy()
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,passenger_id,constant_col,embarked_town_dirty
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1,1,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,1,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3,1,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,4,1,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,5,1,S


# **Data quality checklist**

Quick checklist for this video:
1. Basic dataset overview
2. Missing values summary
3. Duplicates
4. Data type validation
5. Constant and quasi constant columns
6. ID like columns
7. String inconsistencies
8. High null columns
9. High zero columns (for numeric features)

In [ ]:
print(df.dtypes)

PassengerId              int64
Survived                 int64
Pclass                   int64
Name                    object
Sex                     object
Age                    float64
SibSp                    int64
Parch                    int64
Ticket                  object
Fare                   float64
Cabin                   object
Embarked                object
passenger_id             int64
constant_col             int64
embarked_town_dirty     object
dtype: object


In [ ]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,passenger_id,constant_col
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000,891.0
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208,446.000000,1.0
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429,257.353842,0.0
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000,1.000000,1.0
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400,223.500000,1.0
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200,446.000000,1.0
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000,668.500000,1.0
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200,891.000000,1.0


In [ ]:
df.nunique()

,0
PassengerId,891
Survived,2
Pclass,3
Name,891
Sex,2
Age,88
SibSp,7
Parch,7
Ticket,681
Fare,248


##Missing Values Handle


> here 78% values are missing which are big we can drop it
> But the Age is 20% we can fill with mean,median or



In [ ]:
missingValueCount = df.isna().sum().sort_values(ascending=False)
misssingValuePercentage = (df.isna().mean()*100).sort_values(ascending=False)
missingDf = pd.DataFrame(
    {
        'missingValueCount': missingValueCount,
        'missingValuePercentage': misssingValuePercentage
    }

)
print(missingDf)

                     missingValueCount  missingValuePercentage
Cabin                              687               77.104377
Age                                177               19.865320
Embarked                             2                0.224467
embarked_town_dirty                  2                0.224467
Pclass                               0                0.000000
Survived                             0                0.000000
PassengerId                          0                0.000000
SibSp                                0                0.000000
Sex                                  0                0.000000
Name                                 0                0.000000
Parch                                0                0.000000
Fare                                 0                0.000000
Ticket                               0                0.000000
passenger_id                         0                0.000000
constant_col                         0                0

#Data Type Validation

In [ ]:
expectedDataTypes = {
    'PassengerId': 'int64',
    'Survived': 'int64',
    'Pclass': 'int64',
    'Name': 'object',
    'Sex': 'object',
    'Age':'float64'}
for col,ex in expectedDataTypes.items():
    if col in df.columns:
      actual = df[col].dtype
      print(f"{col} actual: {actual} , expected: {ex}")




PassengerId actual: int64 , expected: int64
Survived actual: int64 , expected: int64
Pclass actual: int64 , expected: int64
Name actual: object , expected: object
Sex actual: object , expected: object
Age actual: float64 , expected: float64


In [ ]:
#Fixing datatypes
df['Pclass'] = df['Pclass'].astype('object')

#Constant And Quasi Constant columns
> **Quasi-constant feature** = A feature where almost all values are the same, with very little variation.

In [ ]:
n_rows = len(df)
print(n_rows)

891


In [ ]:
numUnique = df.nunique()
print(numUnique)

PassengerId            891
Survived                 2
Pclass                   3
Name                   891
Sex                      2
Age                     88
SibSp                    7
Parch                    7
Ticket                 681
Fare                   248
Cabin                  147
Embarked                 3
passenger_id           891
constant_col             1
embarked_town_dirty      3
dtype: int64


In [ ]:
constantCol = numUnique[numUnique == 1 ].index.to_list()
print(constantCol)

['constant_col']


In [ ]:
#Quasi-Constant
quasiCol = []
for col in df.columns:
  topFreq = df[col].value_counts(normalize=True, dropna=False).values[0]
  if topFreq >=.95 and col not in constantCol :
    quasiCol.append(col)
print(quasiCol)

[]


#ID like columns

In [ ]:
idLikeCol = []
n_rows = len(df)
for col in df.columns:
  if df[col].nunique(dropna=False) == n_rows:
    idLikeCol.append(col)
print(idLikeCol)

['PassengerId', 'Name', 'passenger_id']


#String Inconsistencies

In [ ]:
df['embarked_town_dirty'] = df['embarked_town_dirty'].astype(str).str.strip().str.lower().replace('unknown', np.nan)

#High Null Columns

In [ ]:
missingValueThreshold = .40 *100
highNullValues = missingDf[missingDf['missingValuePercentage'] >= missingValueThreshold].index.to_list()
print(highNullValues)

['Cabin']


#High zero columns (numeric)

In [ ]:
numeriCol = df.select_dtypes(include=np.number).columns.to_list()
zeroCol = {}
for col in numeriCol:
  zeroCol[col] = (df[col] == 0).mean()
zeroCol = pd.Series(zeroCol).sort_values(ascending=False)
print(zeroCol)
highZeroThershold = .80
highZeroCol = zeroCol[zeroCol >= highZeroThershold].index.to_list()
print(highZeroCol)

Parch           0.760943
SibSp           0.682379
Survived        0.616162
Fare            0.016835
Age             0.000000
PassengerId     0.000000
passenger_id    0.000000
constant_col    0.000000
dtype: float64
[]


#Data Quality Checks - Summary

In [ ]:
print("Data quality checklist summary")
print("- Shape and columns checked")
print("- Missing values summary created")
print("- Duplicates counted and removable copy created")
print("- Data types compared with expectations")
print("- Constant and quasi constant columns flagged")
print("- ID like columns detected")
print("- String inconsistencies checked and cleaned example")
print("- High null columns identified")
print("- High zero numeric columns identified")

Data quality checklist summary
- Shape and columns checked
- Missing values summary created
- Duplicates counted and removable copy created
- Data types compared with expectations
- Constant and quasi constant columns flagged
- ID like columns detected
- String inconsistencies checked and cleaned example
- High null columns identified
- High zero numeric columns identified
